# ActDiff-VC on Google Colab

Compress your own video with **ActDiff-VC**: [paper](https://openreview.net/pdf?id=zER4e4GfZH) · [project website](https://importamir.github.io/ActDiff-VC/) · [code](https://github.com/importAmir/ActDiff-VC).

**Requirements**
- A Colab GPU runtime with **at least 40 GB of GPU memory** (A100 or H100). These need a paid plan (Colab Pro or pay-as-you-go). The free T4 runtime does **not** have enough memory for the Diffusion-as-Shader decoder.
- About 45 GB of free disk (models ≈ 30 GB, environment ≈ 7 GB).

Select **Runtime → Change runtime type → A100 GPU**, then run the cells in order. The first setup (environment and model downloads) takes roughly 20–30 minutes. Each video then takes a few minutes per 49 frames.

## 1. Check the GPU

In [ ]:
import subprocess

info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                      capture_output=True, text=True)
if info.returncode != 0:
    raise SystemExit("No GPU found. Select Runtime -> Change runtime type -> A100 GPU.")
name, memory_mb = [x.strip() for x in info.stdout.splitlines()[0].split(",")]
print(f"GPU: {name}, {int(memory_mb) / 1024:.0f} GB")
if int(memory_mb) < 38 * 1024:
    raise SystemExit("ActDiff-VC needs a GPU with at least 40 GB of memory (A100 or H100).")

## 2. Install the code and the environment

In [ ]:
%cd /content
!git clone -q https://github.com/importAmir/ActDiff-VC.git
%cd /content/ActDiff-VC
!git submodule update --init -q
!curl -fsSL https://pixi.sh/install.sh | bash
import os
os.environ["PATH"] = f"{os.path.expanduser('~/.pixi/bin')}:{os.environ['PATH']}"
!pixi install

## 3. Download the pretrained models

In [ ]:
%cd /content/ActDiff-VC
!pixi run download-das-checkpoint
!pixi run download-alltracker-checkpoint
!pixi run download-hed-checkpoint
!pixi run download-hific-checkpoints
!pixi run download-i3d-weights

## 4. Upload a video

Upload an `.mp4`, `.avi` or `.mov` file. Videos are processed at 480×720.

In [ ]:
from google.colab import files

uploaded = files.upload()
video_path = f"/content/{next(iter(uploaded))}"
print("Video:", video_path)

## 5. Compress, decode and evaluate

In [ ]:
#@markdown Image codec for the keyframes (`hific-low` gives the lowest bit rate).
keyframe_codec = "hific-low"  #@param ["hific-low", "hific-med", "hific-high"]
#@markdown Longest GOP in frames (2-49). Shorter GOPs send more keyframes: higher quality and bit rate.
max_gop_length = 49  #@param {type:"slider", min:2, max:49, step:1}
#@markdown Compress only the first N frames (0 = whole video). Each 49 frames take a few minutes.
num_frames = 49  #@param {type:"integer"}
#@markdown Diffusion steps (20 is faster; 50 gives slightly lower FID / KID).
diffusion_steps = 20  #@param {type:"slider", min:10, max:50, step:10}

%cd /content/ActDiff-VC
frame_arg = f"--num_frames {num_frames}" if num_frames > 0 else ""
!pixi run compress --video_path "{video_path}" --output_dir /content/results \
    --keyframe_codec {keyframe_codec} --max_gop_length {max_gop_length} \
    --diffusion_steps {diffusion_steps} {frame_arg}

## 6. Results

In [ ]:
import base64, glob, json, os
from IPython.display import HTML, display

run_dir = max(glob.glob("/content/results/*/"), key=os.path.getmtime)
metrics = json.load(open(os.path.join(run_dir, "metrics.json")))
print(f"Run: {run_dir}")
print(f"Bit rate: {metrics['bpp']:.4f} bpp "
      f"(keyframes {100 * metrics['bit_breakdown']['keyframes_pct']:.0f}%, "
      f"trajectories {100 * metrics['bit_breakdown']['trajectories_pct']:.0f}%)")
for key in ["psnr", "ms_ssim", "lpips", "tlpips", "niqe", "fid", "pvcs"]:
    print(f"{key:8s} {metrics[key]:.4f}")
print(f"{'kid':8s} {metrics['kid']['mean']:.4f}")

video = base64.b64encode(open(os.path.join(run_dir, "comparison.mp4"), "rb").read()).decode()
display(HTML(f'<p>Original (left) | ActDiff-VC (right)</p>'
             f'<video width="100%" controls loop><source src="data:video/mp4;base64,{video}" type="video/mp4"></video>'))

In [ ]:
# Download the reconstruction, comparison video and metrics as a zip file.
import shutil
from google.colab import files

archive = shutil.make_archive("/content/actdiff_vc_results", "zip", run_dir)
files.download(archive)